# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL (FAIR^2, DOI: [10.71728/senscience.vq4a-28xa](https://sen.science/doi/10.71728/senscience.vq4a-28xa)).

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL (Croissant schema)
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print("Dataset loaded successfully!\n")
print(f"Dataset Title: {metadata.name}")
print(f"Dataset Description: {metadata.description}")
print(f"Version: {metadata.version}")
print(f"License: {metadata.license}\n")
print(f"Published: {metadata.date_published if hasattr(metadata,'date_published') else getattr(metadata, 'datePublished', '')}")
print(f"Authors: {metadata.author if hasattr(metadata, 'author') else ''}")

## 2. Data Overview
Review available record sets, fields, and their IDs. All references use their unique `@id` values.

In [ ]:
# List available record sets with their @id
print("Available `record_set` IDs:")
record_sets = [r['@id'] for r in metadata.record_set]
for rs in metadata.record_set:
    print(f"- RecordSet @id: {rs['@id']} (name: {rs.get('name','')})")


# Display fields for each record set
for rs in metadata.record_set:
    print(f"\nRecordSet '{rs.get('name','')}' (@id: {rs['@id']}):")
    fields = rs.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    for f in fields:
        print(f"  - Field @id: {f['@id']}, name: {f.get('name')}, dataType: {f.get('dataType')}" )

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis using the record set and field `@id`s from the overview.

In [ ]:
# Prepare for extraction: gather RecordSet @id values
record_set_ids = [rs['@id'] for rs in metadata.record_set]

# Create a dictionary of DataFrames for each RecordSet
dataframes = {}
for record_set_id in record_set_ids:
    print(f"Loading records from RecordSet {record_set_id} ...")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    if not df.empty:
        print(f"Columns in RecordSet {record_set_id}:")
        print(df.columns.tolist())
        print(df.head(2))
        break  # Show only the first non-empty RecordSet for brevity

# Select the main tabular RecordSet for further EDA
main_record_set_id = record_set_ids[0]  # Update if your main data is a different record set
# Show first 5 records
print(f"\nSample records from {main_record_set_id}:")
display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. For this, use only field `@id`s for selection, as in:

- Remove outliers
- Normalize a numeric column
- Group by a categorical attribute

In [ ]:
# Identify numeric fields in the main record set
main_fields = metadata.record_set[0].get('field', [])
if isinstance(main_fields, dict):
    main_fields = [main_fields]

numeric_field_id = None
for f in main_fields:
    # Consider '@id' references
    if f.get('dataType', '').lower() in ['float', 'integer', 'number']:
        if f.get('@id') in dataframes[main_record_set_id].columns:
            numeric_field_id = f['@id']
            print(f"Found numeric field: {numeric_field_id} ({f.get('name','')})")
            break

if not numeric_field_id:
    print("No numeric field available.")
else:
    # Filter
    threshold = dataframes[main_record_set_id][numeric_field_id].mean()
    filtered_df = dataframes[main_record_set_id][dataframes[main_record_set_id][numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > mean ({threshold:.2f}):")
    display(filtered_df.head())

    # Normalization
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Try grouping by the first non-numeric field
    group_field_id = None
    for f in main_fields:
        if f.get('dataType', '').lower() not in ['float', 'integer', 'number'] and f.get('@id') in dataframes[main_record_set_id].columns:
            group_field_id = f['@id']
            break

    if group_field_id:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped mean {numeric_field_id} by {group_field_id}:")
        display(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt

if numeric_field_id:
    plt.figure(figsize=(8,5))
    data = dataframes[main_record_set_id][numeric_field_id].dropna()
    plt.hist(data, bins=30, alpha=0.7)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    if group_field_id:
        plt.figure(figsize=(12,4))
        grouped = dataframes[main_record_set_id].groupby(group_field_id)[numeric_field_id].mean().sort_values()
        grouped.plot.bar()
        plt.title(f"Mean {numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.tight_layout()
        plt.show()

## 6. Conclusion
This notebook demonstrated metadata parsing, record set extraction, and basic analysis for the FAIR^2 mass spectrometry dataset using `mlcroissant`. Key steps included reference to all entities via their `@id`, field normalization, and basic plotting using only information from the schema.

<br>_You can now use these analytical patterns for other data processing and ML workflows by referencing fair and reproducible data via their Croissant `@id`s and standard metadata fields._